# Rossmann Store Sales — Exploratory Data Analysis

## Objective

Explore the cleaned Rossmann data to understand the main drivers and patterns of store sales before feature engineering and modeling.

The analysis will focus on asking business-oriented questions rather than generating plots without a clear purpose.

## Key Questions

- How are daily sales distributed?
- How strongly are sales related to store opening status?
- How do promotions affect sales?
- How do sales vary across days of the week?
- How do store types and assortment types differ in sales behavior?
- How do holidays affect sales?
- What temporal patterns exist in sales?
- Are there important differences between stores?

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

In [ ]:
TRAIN_PATH.exists(), STORE_PATH.exists()

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
store = pd.read_parquet(STORE_PATH)

In [ ]:
train.shape, store.shape

In [ ]:
train["Open"].value_counts(dropna=False)

In [ ]:
train.groupby("Open")["Sales"].describe()

### Open Status and Sales

Closed-store observations consistently have zero sales, which is logically consistent with the `Open` indicator.

However, some observations with `Open = 1` have zero sales. These cases require further investigation before deciding whether they represent valid business situations or unusual records.

In [ ]:
closed_with_sales = ((train["Open"] == 0) & (train["Sales"] > 0)).sum()

open_with_zero_sales = ((train["Open"] == 1) & (train["Sales"] == 0)).sum()

closed_with_sales, open_with_zero_sales

In [ ]:
train.loc[
    (train["Open"] == 1) & (train["Sales"] == 0),
    ["Store", "Date", "Sales", "Customers", "Promo", "StateHoliday", "SchoolHoliday"],
].head(20)

### Investigation: Open Stores With Zero Sales

There are 54 observations where the store is marked as open but recorded zero sales. These observations are investigated to determine whether they share common characteristics.

In [ ]:
open_zero_sales = train.loc[(train["Open"] == 1) & (train["Sales"] == 0)].copy()

In [ ]:
open_zero_sales.shape

In [ ]:
open_zero_sales["Customers"].value_counts(dropna=False)

In [ ]:
open_zero_sales.loc[
    open_zero_sales["Customers"] > 0,
    ["Store", "Date", "Sales", "Customers", "Promo", "StateHoliday", "SchoolHoliday"],
]

In [ ]:
open_zero_sales["Store"].nunique()

In [ ]:
open_zero_sales["Store"].value_counts().head(10)

In [ ]:
open_zero_sales["Date"].min(), open_zero_sales["Date"].max()

In [ ]:
open_zero_sales["Date"].value_counts().head(10)

### Findings: Open Stores With Zero Sales

- 54 observations have `Open = 1` and `Sales = 0`.
- These observations are distributed across 41 different stores and span from 2013-01-17 to 2015-05-15.
- 52 of the 54 observations also have zero customers.
- Two observations have positive customer counts despite zero recorded sales.
- The pattern is not strongly concentrated in a single store or a single time period.
- These observations are retained for now because there is not enough evidence to classify them as erroneous.

In [ ]:
sales_without_customers = ((train["Sales"] > 0) & (train["Customers"] == 0)).sum()

customers_without_sales = ((train["Sales"] == 0) & (train["Customers"] > 0)).sum()

sales_without_customers, customers_without_sales

### Sales-Customer Consistency

- No observations have positive sales with zero recorded customers.
- Only two observations have positive customer counts with zero sales.
- Overall, `Sales` and `Customers` are highly consistent from a business-logic perspective.
- The two unusual observations are retained because there is insufficient evidence to classify them as erroneous.

In [ ]:
open_days = train.loc[train["Open"] == 1].copy()

In [ ]:
open_days["Promo"].value_counts()

In [ ]:
open_days.groupby("Promo")["Sales"].agg(["count", "mean", "median", "std"])

### Promotion and Sales

Among open-store observations, promotional days are associated with substantially higher sales.

- Mean sales increase from approximately 5,929 without promotion to 8,228 with promotion.
- Median sales increase from 5,459 to 7,649.
- Sales quartiles are consistently higher during promotional periods.
- Sales variability is also higher on promotional days.

These results indicate a strong association between promotion and sales, but they do not establish a causal effect because promotions may coincide with particular stores, weekdays, seasons, or other factors.

In [ ]:
open_days.groupby("Promo")["Customers"].agg(["count", "mean", "median"])

In [ ]:
open_days_with_customers = open_days.loc[open_days["Customers"] > 0].copy()

In [ ]:
open_days_with_customers["SalesPerCustomer"] = (
    open_days_with_customers["Sales"] / open_days_with_customers["Customers"]
)

In [ ]:
open_days_with_customers.groupby("Promo")["SalesPerCustomer"].agg(["mean", "median"])

### Why Are Sales Higher During Promotions?

Promotional days are associated with both:

- Higher customer traffic: mean customer count increases from about 697 to 844.
- Higher sales per customer: mean sales per customer increases from about 8.94 to 10.18.

Therefore, the higher sales observed during promotions appear to be associated with both more customers and greater sales per customer.

This is an association, not yet evidence of a causal effect.

In [ ]:
open_days.groupby("DayOfWeek")["Promo"].agg(["count", "mean"])

In [ ]:
open_days.groupby(["DayOfWeek", "Promo"])["Sales"].agg(["count", "mean", "median"])

### Promotion Effect Across Weekdays

Promotions are only observed on weekdays 1 through 5; no promotional observations occur on days 6 or 7.

Within each weekday from 1 to 5, both mean and median sales are consistently higher on promotional days.

This suggests that the overall association between promotion and higher sales is not explained solely by weekday composition. However, the analysis remains observational and does not establish a causal promotion effect.

### Promotion Association Across Stores

The overall promotion-sales relationship may hide substantial differences between individual stores.

We compare each store's average sales on promotional and non-promotional open days to investigate store-level heterogeneity.

In [ ]:
promo_sales_by_store = (
    open_days.groupby(["Store", "Promo"])["Sales"].mean().unstack("Promo")
)

promo_sales_by_store.head()

In [ ]:
promo_sales_by_store = promo_sales_by_store.rename(
    columns={0: "NoPromoMeanSales", 1: "PromoMeanSales"}
)

promo_sales_by_store.head()

In [ ]:
promo_sales_by_store.isna().sum()

In [ ]:
promo_sales_by_store.shape

In [ ]:
(promo_sales_by_store["NoPromoMeanSales"] == 0).sum()

In [ ]:
comparable_stores = promo_sales_by_store.copy()

comparable_stores["SalesDifference"] = (
    comparable_stores["PromoMeanSales"] - comparable_stores["NoPromoMeanSales"]
)

comparable_stores["SalesUpliftPct"] = (
    comparable_stores["SalesDifference"] / comparable_stores["NoPromoMeanSales"] * 100
)

In [ ]:
comparable_stores["SalesUpliftPct"].describe()

In [ ]:
(comparable_stores["SalesUpliftPct"] > 0).sum()

In [ ]:
(comparable_stores["SalesUpliftPct"] <= 0).sum()

In [ ]:
comparable_stores.nlargest(
    10,
    "SalesUpliftPct",
)

In [ ]:
comparable_stores.nsmallest(
    10,
    "SalesUpliftPct",
)

### Store-Level Promotion Findings

- All 1,115 stores have observations for both promotional and non-promotional open days.
- Promotional days are associated with higher average sales in 1,114 of the 1,115 stores.
- The median store-level sales uplift is approximately 40.6%.
- Promotion-associated uplift varies substantially across stores, indicating strong store-level heterogeneity.
- One store, Store 274, shows lower average sales during promotional days.
- These comparisons are observational and may still be influenced by seasonality, weekday patterns, or promotion timing.

In [ ]:
open_days.loc[open_days["Store"] == 274].groupby("Promo")["Sales"].agg(
    ["count", "mean", "median", "std"]
)

In [ ]:
open_days.loc[open_days["Store"] == 274].groupby(["DayOfWeek", "Promo"])["Sales"].agg(
    ["count", "mean", "median"]
)

In [ ]:
open_days.loc[open_days["Store"] == 198].groupby("Promo")["Sales"].agg(
    ["count", "mean", "median", "std"]
)

### Investigation of Store 274

Store 274 initially appeared to have lower average sales during promotions.

However, weekday-level analysis shows that promotional sales are higher on days 2 through 5 and nearly unchanged on day 1.

The negative overall uplift is partly explained by weekday composition: days 6 and 7 never contain promotions, and Store 274 has particularly high non-promotional sales on day 7.

Therefore, the raw store-level comparison is confounded by day-of-week effects.

In [ ]:
weekday_open_days = open_days.loc[open_days["DayOfWeek"].between(1, 5)].copy()

In [ ]:
weekday_promo_by_store = (
    weekday_open_days.groupby(["Store", "Promo"])["Sales"]
    .mean()
    .unstack("Promo")
    .rename(
        columns={
            0: "NoPromoMeanSales",
            1: "PromoMeanSales",
        }
    )
)

In [ ]:
weekday_promo_by_store["SalesUpliftPct"] = (
    (
        weekday_promo_by_store["PromoMeanSales"]
        - weekday_promo_by_store["NoPromoMeanSales"]
    )
    / weekday_promo_by_store["NoPromoMeanSales"]
    * 100
)

In [ ]:
weekday_promo_by_store["SalesUpliftPct"].describe()

In [ ]:
weekday_promo_by_store.loc[274]

In [ ]:
(weekday_promo_by_store["SalesUpliftPct"] <= 0).sum()

### Weekday-Adjusted Store-Level Promotion Findings

After restricting the comparison to weekdays 1 through 5, all 1,115 stores show higher average sales on promotional days.

- Mean store-level uplift: approximately 40.0%.
- Median store-level uplift: approximately 39.5%.
- Minimum uplift: approximately 1.9%.
- Maximum uplift: approximately 102.1%.
- No store shows non-positive uplift after removing weekend composition effects.

Store 274 changes from a negative raw uplift to approximately +8.7%, confirming that its initial result was confounded by weekday composition.

The promotion-sales relationship remains observational and should not be interpreted as causal.

In [ ]:
eda = train.merge(store, on="Store", how="left", validate="many_to_one")

In [ ]:
eda.shape

In [ ]:
eda["StoreType"].isna().sum()

In [ ]:
open_eda = eda.loc[eda["Open"] == 1].copy()

open_eda.groupby("StoreType")["Sales"].agg(["count", "mean", "median", "std"])

### Sales by Store Type

Among open-store observations, StoreType `b` shows substantially higher mean and median sales than the other store types.

StoreTypes `a`, `c`, and `d` have relatively similar average sales levels, while type `b` also shows greater sales variability.

These differences are descriptive and should not be interpreted as a causal effect of store type.

In [ ]:
open_eda.groupby("StoreType")["Customers"].agg(["count", "mean", "median"])

In [ ]:
open_eda_with_customers = open_eda.loc[open_eda["Customers"] > 0].copy()

open_eda_with_customers["SalesPerCustomer"] = (
    open_eda_with_customers["Sales"] / open_eda_with_customers["Customers"]
)

In [ ]:
open_eda_with_customers.groupby("StoreType")["SalesPerCustomer"].agg(["mean", "median"])

### Customer Behavior by Store Type

StoreType `b` has substantially higher daily customer traffic than the other store types, which explains much of its higher total sales.

However, StoreType `b` has the lowest sales per customer, while StoreType `d` has the highest sales per customer despite having the lowest average customer traffic.

This suggests that store types differ not only in total sales, but also in the way those sales are generated:
- Type `b`: high traffic, lower sales per customer.
- Type `d`: lower traffic, higher sales per customer.
- Types `a` and `c`: relatively similar customer and spending patterns.

In [ ]:
store["StoreType"].value_counts()

In [ ]:
pd.crosstab(
    store["StoreType"],
    store["Assortment"],
    normalize="index",
) * 100

### Store Type and Assortment Relationship

Store type and assortment are not distributed independently.

- StoreType `a` is mostly associated with Assortment `a`.
- StoreType `d` is mostly associated with Assortment `c`.
- Assortment `b` appears only within StoreType `b`.
- StoreType `b` contains only 17 stores, so comparisons involving this group should be interpreted cautiously.

This relationship means that differences in sales across store types may partly reflect differences in assortment composition.

In [ ]:
open_eda.groupby("Assortment")["Sales"].agg(["count", "mean", "median", "std"])

In [ ]:
open_eda.groupby(["StoreType", "Assortment"])["Sales"].agg(["count", "mean", "median"])

### Sales by Store Type and Assortment

Aggregate sales differ across assortment categories, but assortment composition is strongly related to store type.

Within StoreTypes `a` and `d`, Assortment `c` is associated with higher average sales than Assortment `a`. The difference is smaller within StoreType `c`.

StoreType `b` shows large differences across assortment categories, but these results should be interpreted cautiously because some StoreType-Assortment combinations contain very few stores.

Therefore, aggregate assortment comparisons may be confounded by store type.

In [ ]:
store_level_sales = (
    open_eda.groupby(["Store", "StoreType", "Assortment"])["Sales"]
    .mean()
    .reset_index(name="MeanSales")
)

store_level_sales.head()

In [ ]:
store_level_sales.groupby(["StoreType", "Assortment"])["MeanSales"].agg(
    ["count", "mean", "median"]
)